<a href="https://colab.research.google.com/github/ababacaryoro/nlp-course/blob/main/lab2/lab2_preprocessing_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 – Text preprocessing pipeline

**Objectives**
- Tokenize text into words and sentences with NLTK and spaCy
- Normalize words with stemming and lemmatization
- Remove stop words and special characters
- Assemble the steps into a reusable preprocessing pipeline

**Instructions**
1. Complete every cell that contains `# YOUR CODE HERE`.
2. Run the cell: each exercise ends with an `assert` that checks the answer.
3. When everything passes, follow the *Submission* section at the end.

## Setup

In [ ]:
# Google Colab only: install the libraries that Colab does not ship (about 1 minute).
# Uncomment the next line and run the cell. Nothing to do for a local installation made with `uv sync`.
# !pip install -q -r https://raw.githubusercontent.com/ababacaryoro/nlp-course/main/requirements.txt

In [ ]:
import re
import string
import nltk
import spacy

# NLTK resources (downloaded once, then cached)
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet"]:
    nltk.download(resource, quiet=True)

# spaCy English model (installed with the course environment / requirements.txt)
nlp = spacy.load("en_core_web_sm")

print("Setup complete")

---
# Part 1 – Tokenization

Tokenization splits a text into smaller units (tokens): words, sub-words, characters or sentences.

## 1.1 Naive tokenization with `split()`

In [ ]:
text = "Natural Language Processing is fascinating!"
tokens_basic = text.split()
print("split():", tokens_basic)
print("Last token:", tokens_basic[-1])   # the punctuation stays attached to the word

## 1.2 Tokenization with NLTK

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

text = "Hello! How are you doing today? I'm learning NLP. It's really interesting."

print("Word tokens:", word_tokenize(text))
print("\nSentence tokens:")
for i, sent in enumerate(sent_tokenize(text), 1):
    print(f"  {i}. {sent}")

In [ ]:
# NLTK separates punctuation and contractions
text = "I can't believe it's not butter! Don't you think so?"
print("split():       ", text.split())
print("word_tokenize():", word_tokenize(text))

In [ ]:
# Exercise 1.1 – Tokenize the text into words with NLTK and count the tokens.
text = "Dr. Smith's patients can't understand why they're feeling unwell. Is it the flu?"

tokens = None      # YOUR CODE HERE
num_tokens = None  # YOUR CODE HERE

print("Tokens:", tokens)
print("Number of tokens:", num_tokens)
assert num_tokens == 18, f"Expected 18 tokens, got {num_tokens}"

## 1.3 Tokenization with spaCy

In [ ]:
text = "Apple is looking at buying U.K. startup for $1 billion."
doc = nlp(text)

print("Tokens:", [token.text for token in doc])
print("\nToken attributes:")
for token in doc:
    print(f"  {token.text:12} | POS: {token.pos_:6} | is_stop: {token.is_stop}")

In [ ]:
# Exercise 1.2 – With spaCy, keep the tokens that are neither punctuation nor whitespace.
# Hint: token.is_punct and token.is_space
text = "The quick, brown fox jumps over the lazy dog! Isn't it amazing?"
doc = nlp(text)

clean_tokens = None  # YOUR CODE HERE (list comprehension)

print("Clean tokens:", clean_tokens)
assert len(clean_tokens) == 13, f"Expected 13 tokens, got {len(clean_tokens)}"

## 1.4 Other tokenizers

In [ ]:
from nltk.tokenize import RegexpTokenizer, TweetTokenizer

text = "Hello! How's it going? #NLP @user123"

# RegexpTokenizer: tokens are defined by a pattern (here, word characters only)
print("Regexp tokens:", RegexpTokenizer(r"\w+").tokenize(text))

# TweetTokenizer: keeps hashtags, mentions and emoticons together
print("Tweet tokens: ", TweetTokenizer().tokenize(text))

In [ ]:
# Exercise 1.3 – Build a RegexpTokenizer that keeps alphabetic words only (no digit, no punctuation).
# Hint: the pattern [a-zA-Z]+ matches one or more letters.
text = "I have 3 cats and 2 dogs! Their names are Max123 and Bella."

alpha_tokenizer = None  # YOUR CODE HERE
alpha_tokens = None     # YOUR CODE HERE

print("Alphabetic tokens:", alpha_tokens)
assert alpha_tokens == ["I", "have", "cats", "and", "dogs", "Their", "names", "are", "Max", "and", "Bella"], "Check your pattern"

---
# Part 2 – Stemming and lemmatization

Both reduce a word to a base form:
- **Stemming** chops the end of the word with rules (fast, may produce non-words).
- **Lemmatization** uses a vocabulary and the part of speech (slower, returns real words).

## 2.1 Stemming with NLTK

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer

porter = PorterStemmer()
words = ["running", "runs", "ran", "runner", "easily", "fairly"]

print("Porter stemmer:")
for word in words:
    print(f"  {word:12} -> {porter.stem(word)}")

In [ ]:
snowball = SnowballStemmer("english")   # also available for other languages
print("Snowball stemmer:")
for word in words:
    print(f"  {word:12} -> {snowball.stem(word)}")
print("\nLanguages:", SnowballStemmer.languages)

In [ ]:
# Stems are not always real words
for word in ["studies", "studying", "university", "universe", "beautiful", "beauty"]:
    print(f"  {word:12} -> {porter.stem(word)}")

In [ ]:
# Exercise 2.1 – Tokenize the sentence with word_tokenize, then stem every token with the Porter stemmer.
sentence = "The cats are running and jumping over the sleeping dogs"

tokens = None          # YOUR CODE HERE
stemmed_tokens = None  # YOUR CODE HERE (list comprehension)

print("Original:", tokens)
print("Stemmed: ", stemmed_tokens)
assert stemmed_tokens == ["the", "cat", "are", "run", "and", "jump", "over", "the", "sleep", "dog"], "Check your stemming"

## 2.2 Lemmatization with NLTK (WordNet)

In [ ]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

words = ["running", "runs", "ran", "better", "studies", "feet", "geese"]
print("Default (every word treated as a noun):")
for word in words:
    print(f"  {word:12} -> {lemmatizer.lemmatize(word)}")

In [ ]:
# The part of speech changes the result: n = noun, v = verb, a = adjective, r = adverb
print("running (verb):    ", lemmatizer.lemmatize("running", pos="v"))
print("running (noun):    ", lemmatizer.lemmatize("running", pos="n"))
print("better (adjective):", lemmatizer.lemmatize("better", pos="a"))
print("studies (verb):    ", lemmatizer.lemmatize("studies", pos="v"))
print("studies (noun):    ", lemmatizer.lemmatize("studies", pos="n"))

In [ ]:
# Exercise 2.2 – Give the right POS tag ("n", "v", "a" or "r") to each word so that the lemma is the expected one.
words_with_pos = [
    ("flying", "v"),    # -> fly
    ("happily", None),  # -> happily      YOUR CODE HERE
    ("worse", None),    # -> bad          YOUR CODE HERE
    ("mice", None),     # -> mouse        YOUR CODE HERE
    ("are", None),      # -> be           YOUR CODE HERE
]

results = [lemmatizer.lemmatize(w, pos=p) for w, p in words_with_pos]
for (w, p), lemma in zip(words_with_pos, results):
    print(f"  {w:12} ({p}) -> {lemma}")
assert results == ["fly", "happily", "bad", "mouse", "be"], f"Got {results}"

## 2.3 Lemmatization with spaCy

In [ ]:
# spaCy detects the part of speech itself
text = "The children are playing with their toys. They were running and jumping happily."
for token in nlp(text):
    if token.text != token.lemma_:
        print(f"  {token.text:12} ({token.pos_:5}) -> {token.lemma_}")

In [ ]:
# Exercise 2.3 – With spaCy, return the lowercase lemma of every token that is not punctuation.
text = "The dogs were barking loudly at the cats who were climbing the trees."
doc = nlp(text)

lemmas = None  # YOUR CODE HERE (list comprehension)

print("Lemmas:", lemmas)
assert lemmas == ["the", "dog", "be", "bark", "loudly", "at", "the", "cat", "who", "be", "climb", "the", "tree"], "Check your lemmatization"

## 2.4 Stemming vs lemmatization

In [ ]:
words = ["studies", "studying", "better", "feet", "ran", "easily", "fairly", "wolves"]
print(f"{'Word':<12} {'Stem':<12} {'Lemma (noun)':<12}")
print("-" * 36)
for word in words:
    print(f"{word:<12} {porter.stem(word):<12} {lemmatizer.lemmatize(word):<12}")

**Takeaways**
- Stemming is fast but produces non-words (`studi`, `easili`).
- Lemmatization returns dictionary words but needs the part of speech to be accurate.
- Stemming suits search engines and simple classifiers; lemmatization suits tasks where the output is read by humans or by language models.

---
# Part 3 – Stop words and special characters

Stop words are very frequent words that carry little meaning on their own (*the, is, at, which, ...*).

## 3.1 Stop words with NLTK

In [ ]:
from nltk.corpus import stopwords
stop_words_nltk = set(stopwords.words("english"))

print(f"Number of NLTK stop words: {len(stop_words_nltk)}")
print(f"Sample: {sorted(stop_words_nltk)[:20]}")

In [ ]:
text = "The quick brown fox jumps over the lazy dog in the park"
tokens = word_tokenize(text.lower())
filtered_tokens = [t for t in tokens if t not in stop_words_nltk]
print("Original:          ", tokens)
print("Without stop words:", filtered_tokens)

In [ ]:
# Exercise 3.1 – Lowercase and tokenize the text, then remove the NLTK stop words.
text = "This is a sample sentence showing the removal of stop words from the text"

tokens = None    # YOUR CODE HERE
filtered = None  # YOUR CODE HERE

print("Filtered tokens:", filtered)
assert filtered == ["sample", "sentence", "showing", "removal", "stop", "words", "text"], "Check your filtering"

## 3.2 Stop words with spaCy

In [ ]:
text = "This is a sample sentence for demonstrating stop word removal."
doc = nlp(text)
for token in doc:
    print(f"  {token.text:<15} is_stop: {token.is_stop}")

content_words = [token.text for token in doc if not token.is_stop and not token.is_punct]
print("\nContent words:", content_words)

In [ ]:
# The spaCy stop list can be customized
nlp.vocab["sample"].is_stop = True    # add a word
nlp.vocab["not"].is_stop = False      # keep "not": it carries meaning for sentiment analysis
print(f"Number of spaCy stop words: {len(nlp.Defaults.stop_words)}")

## 3.3 Punctuation and special characters

In [ ]:
print("string.punctuation:", string.punctuation)

# Method 1: str.translate()
text = "Hello, World! How's it going? #NLP @user"
translator = str.maketrans("", "", string.punctuation)
print("translate():", text.translate(translator))

In [ ]:
# Method 2: regular expressions
text = "Hello, World! How's it going? #NLP @user 123"
print("Letters only:      ", re.sub(r"[^a-zA-Z\s]", "", text))
print("Letters and digits:", re.sub(r"[^\w\s]", "", text))

In [ ]:
# Method 3: spaCy token attributes
text = "Hello! This is @user's tweet about #NLP. Check https://example.com!"
doc = nlp(text)
tokens_spacy = [token.text for token in doc if not token.is_punct and not token.is_space and not token.like_url]
print("Clean tokens:", tokens_spacy)

In [ ]:
# Exercise 3.2 – Clean the text with regular expressions, in this order:
#   1. remove URLs
#   2. remove mentions: "@OpenAI's" must disappear entirely, use @\S+ (not @\w+)
#   3. remove hashtags
#   4. remove punctuation
#   5. lowercase
#   6. normalize whitespace
text = "Check out @OpenAI's new model! https://openai.com #AI #MachineLearning It's amazing!!!"

clean = text
clean = None  # YOUR CODE HERE: remove URLs
clean = None  # YOUR CODE HERE: remove mentions
clean = None  # YOUR CODE HERE: remove hashtags
clean = None  # YOUR CODE HERE: remove punctuation
clean = None  # YOUR CODE HERE: lowercase
clean = None  # YOUR CODE HERE: normalize whitespace

print(f"Clean text: '{clean}'")
assert clean == "check out new model its amazing", f"Got: '{clean}'"

---
# Part 4 – A complete preprocessing pipeline

## 4.1 Example pipeline

In [ ]:
def preprocess_pipeline_example(text):
    """Lowercase, remove URLs and special characters, tokenize with spaCy, remove stop words, lemmatize."""
    text = text.lower()
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    doc = nlp(text)
    return [token.lemma_ for token in doc
            if not token.is_stop and not token.is_punct and not token.is_space and len(token.text) > 1]

sample_text = """
The quick brown foxes are jumping over the lazy dogs! 
Check out https://example.com for more information.
This is SO amazing!!! #NLP #Python @user123
"""
print(preprocess_pipeline_example(sample_text))

## 4.2 Final challenge: build your own pipeline

In [ ]:
# Exercise 4.1 – Write `preprocess_text(text)` that returns a list of tokens after:
#   1. lowercase
#   2. remove URLs
#   3. remove e-mail addresses
#   4. remove mentions and hashtags
#   5. remove numbers
#   6. remove punctuation and special characters
#   7. tokenize with spaCy
#   8. remove stop words
#   9. lemmatize
#  10. remove tokens shorter than 2 characters
def preprocess_text(text):
    """Complete preprocessing pipeline. Returns a list of tokens."""
    # YOUR CODE HERE
    pass

In [ ]:
test_text = """
🚀 BREAKING NEWS!!! The researchers at @MIT have published 5 new papers 
about Natural Language Processing! Check out https://mit.edu/nlp for details.
Contact them at research@mit.edu for collaborations. #NLP #AI #Research

The experiments were conducted using state-of-the-art transformers models.
They achieved 95.5% accuracy on the benchmark datasets!!!
"""

result = preprocess_text(test_text)
print(result)

for token in ["researcher", "publish", "paper", "natural", "language", "processing"]:
    assert token in result, f"Expected '{token}' in the result"
for item in ["@mit", "https", "mit.edu", "#nlp", "95.5", "!!!", "the", "a", "at"]:
    assert item not in result, f"'{item}' should not be in the result"
print("\nAll tests passed")

## 4.3 Applying the pipeline to several documents

In [ ]:
# Exercise 4.2 – Apply your pipeline to each document (list of lists of tokens).
documents = [
    "Machine learning is transforming the tech industry! @Google #ML",
    "I love programming in Python. It's so easy to learn! https://python.org",
    "The cats are sleeping on the couch. They're so lazy!",
    "Contact support@company.com for any questions about our AI products."
]

processed_docs = None  # YOUR CODE HERE (list comprehension)

for i, doc in enumerate(processed_docs, 1):
    print(f"  Doc {i}: {doc}")
assert len(processed_docs) == 4 and "python" in processed_docs[1], "Check your pipeline"

---
## Summary

| Step | NLTK | spaCy |
|------|------|-------|
| Word / sentence tokenization | `word_tokenize()`, `sent_tokenize()`, `RegexpTokenizer`, `TweetTokenizer` | `nlp(text)` |
| Stemming | `PorterStemmer`, `SnowballStemmer` | – |
| Lemmatization | `WordNetLemmatizer` (needs the POS) | `token.lemma_` (POS detected automatically) |
| Stop words | `stopwords.words("english")` | `token.is_stop` |
| Punctuation | `string.punctuation`, `re.sub()` | `token.is_punct`, `token.like_url` |

Order matters: lowercase before the regular expressions, tokenize before lemmatizing, and decide which stop words to keep for the task (for example `not` for sentiment analysis).

---

## Submission

1. Run the whole notebook from top to bottom (`Kernel > Restart & Run All`) and check that no cell fails.
2. Keep the outputs, save the notebook and commit it in your course repository (`nlp-course-LASTNAME`), in the same folder as in the course repository.
3. Push, then send an e-mail to the teacher to say that the work is ready. The link to the repository is only needed in your first e-mail.